# float32, posit32, posit16 i float64 na tych samych algorytmach

Etap 3 planu: pierwszy eksperyment. Funkcje są w `examples/formats.py`; tutaj wykresy.
Na KV260 posit32 może liczyć sprzęt: w drugiej komórce `USE_HW = True`
(wymaga overlayu `posit32_vec`, `sudo` dla `/dev/uio*`).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))          # sw/
sys.path.insert(0, str(pathlib.Path.cwd().parent / "examples"))
import numpy as np, matplotlib.pyplot as plt
import positnp as pn
import formats

In [ ]:
USE_HW = False
if USE_HW:
    from positnp.backend_hw import PositVecHw
    pn.set_backend(PositVecHw.open(32))
pn.clear_stats()

## 1. Horner: (x − 1)⁷ rozpisane na wielomian

Współczynniki do 35, wynik od 2·10⁻⁷ do 1: każdy krok traci cyfry na znoszeniu się składników.
Oś Y: liczba poprawnych cyfr dziesiętnych względem dokładnego (x − 1)⁷.

In [ ]:
x, exact, rows = formats.experiment_horner()
fig, ax = plt.subplots(figsize=(9, 4))
for name, d in rows.items():
    ax.plot(x, d, label=name, lw=1)
ax.set_xlabel("x"); ax.set_ylabel("poprawne cyfry"); ax.legend(); ax.grid(alpha=.3)
{name: round(float(np.median(d)), 2) for name, d in rows.items()}

## 2. Suma 1/k drzewem (bez quire)

log₂ N dodawań połówek tablicy. Dokładną sumę z jednym zaokrągleniem da quire w etapie 4.

In [ ]:
res = formats.experiment_sum()
{name: (s, round(float(formats.digits(np.array([s]), ex)[0]), 2)) for name, (s, ex) in res.items()}

In [ ]:
pn.stats()   # flagi z backendu posit32: ile wyników niedokładnych, nasyconych, NaR